# 短期反转因子回测研究

本notebook演示短期反转因子（5日收益 - 1日收益）的完整回测流程。

## 因子逻辑
- **5日收益率**：短期趋势方向
- **1日收益率**：最近一天的反转信号
- **因子值 = 5日收益 - 1日收益**：剥离最近一天扰动，捕捉反转机会

In [ ]:
import sys
import os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.data_loader import DataLoader
from src.factors.momentum import ShortTermReversalFactor
from src.utils.helpers import load_config

plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False
sns.set_style('whitegrid')

print('环境准备完成')

## 1. 加载配置与数据

In [ ]:
# 加载配置
config = load_config(os.path.join('..', 'config', 'config.yaml'))
print(f"股票池: {config['data']['universe']}")
print(f"时间范围: {config['data']['start_date']} ~ {config['data']['end_date']}")

# 加载数据
loader = DataLoader(
    raw_dir=config['data']['raw_dir'],
    processed_dir=config['data']['processed_dir']
)
data = loader.load_data(
    start_date=config['data']['start_date'],
    end_date=config['data']['end_date']
)
print(f"\n数据形状: {data.shape}")
data.head()

## 2. 计算短期反转因子

In [ ]:
# 初始化因子
factor = ShortTermReversalFactor(
    trend_window=config['factor']['trend_window'],
    reversal_window=config['factor']['reversal_window']
)

# 计算因子值并分组
factor_data = factor.compute_with_groups(
    data,
    num_groups=config['backtest']['num_groups'],
    normalize_method='zscore'
)

print(f"因子计算完成: {factor_data.shape}")
print("\n因子值统计:")
factor_data['factor_value'].describe()

## 3. 因子分析

In [ ]:
# 分组收益分析
group_returns = factor_data.groupby('group')['daily_return'].mean() * 252

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 各组年化收益
axes[0].bar(group_returns.index, group_returns.values, color='steelblue')
axes[0].set_title('各组年化收益率')
axes[0].set_xlabel('分组 (1=最高分)')
axes[0].set_ylabel('年化收益率')

# 因子值分布
axes[1].hist(factor_data['factor_value'], bins=50, color='coral', alpha=0.7)
axes[1].set_title('因子值分布')
axes[1].set_xlabel('因子值')
axes[1].set_ylabel('频数')

plt.tight_layout()
plt.savefig('../results/plots/factor_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n各组年化收益:')
print(group_returns)

## 4. 回测结果

运行完整回测请参考 `scripts/run_backtest.py`

In [ ]:
# 多空组合收益
long_return = group_returns.loc[1]  # Top组
short_return = group_returns.loc[5]  # Bottom组
long_short = long_return - short_return

print(f"Top组年化收益: {long_return:.2%}")
print(f"Bottom组年化收益: {short_return:.2%}")
print(f"多空收益: {long_short:.2%}")